In [33]:
# Libraries: 
import tensorflow as tf
import onnx
import tf2onnx
import numpy as np
import json
import pickle
import os

print("TF version:", tf.__version__)
print("tf2onnx:", tf2onnx.__version__)

TF version: 2.15.0
tf2onnx: 1.16.1


In [35]:
ART_DIR = "../artifacts/latest"

with open(os.path.join(ART_DIR, "feature_config.json")) as f:
    feature_config = json.load(f)

sequence_length = feature_config["sequence_length"]
num_features = feature_config["num_features"]
dim_weights = feature_config["dim_weights"]

In [37]:
num_features

5

In [38]:
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, RepeatVector, Dense, TimeDistributed
from tensorflow.keras.models import Model

class LSTMTimeSeriesAutoencoder:
    def __init__(self, sequence_length, num_dimensions, dim_weights):
        self.sequence_length = sequence_length
        self.num_dimensions = num_dimensions
        self.dim_weights = dim_weights
        self.model = self.build_model()

    def build_model(self):
        # Define input shape
        input_shape = (self.sequence_length, self.num_dimensions)

        # Encoder
        encoder_inputs = Input(shape=input_shape)
        encoder = LSTM(128, return_sequences=True)(encoder_inputs)
        encoder = LSTM(64, return_sequences=True)(encoder)
        encoder = LSTM(32, return_sequences=True)(encoder)
        encoder = LSTM(16, return_sequences=True)(encoder)
        encoder = LSTM(8, return_sequences=False)(encoder)

        # Repeat the latent vector for each time step
        decoder_inputs = RepeatVector(self.sequence_length)(encoder)

        # Decoder
        decoder = LSTM(8, return_sequences=True)(decoder_inputs)
        decoder = LSTM(16, return_sequences=True)(decoder)
        decoder = LSTM(32, return_sequences=True)(decoder)
        decoder = LSTM(64, return_sequences=True)(decoder)
        decoder = LSTM(128, return_sequences=True)(decoder)

        # Output layer
        output = TimeDistributed(Dense(self.num_dimensions))(decoder)

        # Define the model
        model = Model(inputs=encoder_inputs, outputs=output)

        return model

    def custom_loss(self, y_true, y_pred):
        # Define weights for each dimension
        weights = tf.constant(self.dim_weights, dtype=tf.float32)

        # Calculate mean squared error (MSE) for each dimension
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=0)

        # Multiply MSE by weights and sum across dimensions
        weighted_loss = tf.reduce_sum(mse * weights)

        return weighted_loss

    def compile_model(self):
        self.model.compile(optimizer='adam', loss=self.custom_loss)

    def summary(self):
        self.model.summary()

    def train(self, x_train, epochs, batch_size, callbacks):
        history = self.model.fit(x_train, x_train, epochs=epochs, batch_size=batch_size, verbose=1, validation_split=0.1,  callbacks=callbacks, shuffle=False)
        return history

    def predict(self, x):
        return self.model.predict(x)

    def save_weights(self, filepath):
        # Save model weights
        self.model.save_weights(filepath)

    def load_weights(self, filepath):
        # Load model weights
        self.model.load_weights(filepath)


In [39]:
model = LSTMTimeSeriesAutoencoder(sequence_length= sequence_length, num_dimensions=num_features, dim_weights = dim_weights)

In [41]:
weights_path = "../artifacts/latest/models/LSTM_Autoencoder_30_April_2024_program_level.keras"

model.load_weights(weights_path)
print("Weights loaded successfully")

Weights loaded successfully


In [42]:
calib_X = np.load(os.path.join(ART_DIR, "training_set.npy"))

pred_tf = model.predict(calib_X[:50])
print("Sanity output shape:", pred_tf.shape)

2/2 [==============================] - 3s 17ms/step
Sanity output shape: (50, 10, 5)


In [43]:
calib_X[10:20,4,0]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [44]:
pred_tf[10:20,4,0]

array([-0.0021989 , -0.0022567 , -0.00233546, -0.00237124, -0.00239661,
       -0.00239758, -0.00238928, -0.00424968, -0.00490938, -0.00604258],
      dtype=float32)

In [45]:
# Results match so far. 

# =================================
# Only one issue, 
# Needed to re-write the LSTM class here. how will this process be replicated during inference. 

In [46]:
type(model)

__main__.LSTMTimeSeriesAutoencoder

In [47]:
type(model.model)

keras.src.engine.functional.Functional

In [48]:
model.model.inputs

[<KerasTensor: shape=(None, 10, 5) dtype=float32 (created by layer 'input_2')>]

In [49]:
model.model.outputs

[<KerasTensor: shape=(None, 10, 5) dtype=float32 (created by layer 'time_distributed_1')>]

In [53]:
actual_model = model.model

spec = (
    tf.TensorSpec(
        (None, sequence_length, num_features),
        tf.float32,
        name="input_sequence"
    ),
)

onnx_model, _ = tf2onnx.convert.from_keras(
    actual_model,
    input_signature=spec,
    opset=15,
    output_path="../artifacts/latest/models/model.onnx"
)

print("ONNX model saved")

ONNX model saved


In [16]:
# Inference Test: 

In [55]:
import onnxruntime as ort

# Start the onnx runtime session: 
session = ort.InferenceSession(
    "../artifacts/latest/models/model.onnx",
    providers=["CPUExecutionProvider"]
)

for inp in session.get_inputs():
    print("Name:", inp.name)
    print("Shape:", inp.shape)
    print("Type:", inp.type)
    print()

Name: input_sequence
Shape: ['unk__1167', 10, 5]
Type: tensor(float)



In [56]:
# Get inputs ready to pass to onnx session: 
calib_X = np.load("../artifacts/latest/training_set.npy")

onnx_input = calib_X[:50].astype(np.float32)
print(onnx_input.shape)

(50, 10, 5)


In [57]:
# Pass input to onnx session and run the prediction
input_name = session.get_inputs()[0].name
pred_onnx = session.run(
    None,
    {input_name: onnx_input}
)[0]

In [58]:
# Get the prediction form original model format and show the difference between both the models' preidcitons: 
pred_tf = model.predict(calib_X[:50])

diff = np.mean(np.abs(pred_tf - pred_onnx))
print("Mean abs difference TF vs ONNX:", diff)

2/2 [==============================] - 0s 6ms/step
Mean abs difference TF vs ONNX: 3.2487513e-08


In [59]:
pred_onnx[10:20,4,0]

array([-0.00219881, -0.00225668, -0.00233538, -0.00237123, -0.00239655,
       -0.00239751, -0.00238922, -0.00424969, -0.00490933, -0.00604259],
      dtype=float32)